# NTU Lecture Hall Electricity — Data Loading & Cleaning

**Project:** Quantifying excess AC electricity from over-cooling in NTU lecture halls  
**Role:** Energy Data Consulting firm presenting to NTU  
**Data:** Hourly electricity meters for 普通, 共同, 博雅, 新生 lecture halls (2016–2025)

### Meters
| Variable | Meter | Coverage |
|---|---|---|
| `df_ac` | 普通高壓空調 (AC sub-meter only) | 2016–2025, ground truth for AC consumption |
| `df_gongtong` | 共同教室 (total electricity) | 2016–2025 |
| `df_boya` | 博雅館一/二/三/四 (total electricity, 4 sub-meters) | 2016–2025 |
| `df_xinsheng` | 新生大樓 (total electricity) | 2016–2025 |

## 0. Setup

In [ ]:
# Install required packages if needed
# !pip install python-calamine beautifulsoup4 html5lib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_DIR = Path('data')

COLS = ['datetime', 'kw', 'meter_val', 'kwh', 'pf',
        'I_r', 'I_s', 'I_t', 'V_rs', 'V_st', 'V_tr', 'kva', 'kvar']

print('Data directory exists:', DATA_DIR.exists())
print('Files found:', len(list(DATA_DIR.glob('*.xls'))))

## 1. File Loading Helpers

Two file formats exist in the data folder:
- **HTML-disguised XLS** (~7.5 MB): most files — use `pd.read_html` with Big5 encoding
- **True binary XLS** (~1.2 MB): 新生大樓 files — use `pd.read_excel` with `engine='calamine'`

In [ ]:
def load_html_xls(path: Path) -> pd.DataFrame:
    """Load an HTML-disguised .xls file (most meters)."""
    tables = pd.read_html(path, encoding='big5')
    # Table 0 is a header block; table 1 has the data with row 0 as column names
    df = tables[1].iloc[1:].copy()
    df.columns = COLS
    return df


def load_binary_xls(path: Path) -> pd.DataFrame:
    """Load a true binary .xls file (新生大樓)."""
    df = pd.read_excel(path, engine='calamine', header=0)
    df = df.iloc[1:].copy()  # row 0 is column names in the sheet
    df.columns = COLS
    return df


def is_binary_xls(path: Path) -> bool:
    """Binary XLS files start with the OLE2 magic bytes d0cf11e0."""
    with open(path, 'rb') as f:
        return f.read(4) == b'\xd0\xcf\x11\xe0'


def load_xls(path: Path) -> pd.DataFrame:
    """Dispatch to the correct loader based on file format."""
    if is_binary_xls(path):
        return load_binary_xls(path)
    return load_html_xls(path)


def clean(df: pd.DataFrame, meter_name: str) -> pd.DataFrame:
    """
    Parse types, remove impossible values, add time features.
    Returns a clean DataFrame indexed by datetime.
    """
    df = df.copy()

    # --- parse types ---
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    for col in ['kw', 'kwh', 'pf', 'meter_val', 'kva', 'kvar',
                'I_r', 'I_s', 'I_t', 'V_rs', 'V_st', 'V_tr']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['datetime']).set_index('datetime').sort_index()

    # --- remove impossible readings ---
    n_before = len(df)
    df = df[df['kw'] >= 0]                        # no negative power
    df = df[df['kwh'] >= 0]                        # no negative energy
    df = df[df['kw'] < df['kw'].quantile(0.9999)]  # remove extreme spikes
    n_after = len(df)
    if n_before != n_after:
        print(f'  [{meter_name}] Removed {n_before - n_after} anomalous rows '
              f'({(n_before - n_after) / n_before * 100:.2f}%)')

    # --- detect & flag meter resets (sudden large drop in cumulative meter_val) ---
    df['meter_reset'] = df['meter_val'].diff() < -1000
    n_resets = df['meter_reset'].sum()
    if n_resets > 0:
        print(f'  [{meter_name}] Flagged {n_resets} potential meter reset(s)')

    # --- time features ---
    df['year']       = df.index.year
    df['month']      = df.index.month
    df['hour']       = df.index.hour
    df['weekday']    = df.index.weekday          # 0=Mon, 6=Sun
    df['is_weekend'] = df['weekday'] >= 5
    df['season']     = df['month'].map({
        12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring',  4: 'Spring', 5: 'Spring',
        6: 'Summer',  7: 'Summer', 8: 'Summer',
        9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
    })
    # NTU academic semesters: ~Sep–Jan (fall) and ~Feb–Jun (spring)
    df['is_semester'] = df['month'].isin([2, 3, 4, 5, 6, 9, 10, 11, 12])

    df['meter'] = meter_name
    return df


def load_meter(prefix: str, meter_name: str,
               years: range = range(2016, 2026)) -> pd.DataFrame:
    """
    Load and concatenate all annual files for a given meter prefix.
    Skips missing files with a warning.
    """
    frames = []
    for yr in years:
        path = DATA_DIR / f'{prefix}{yr}.xls'
        if not path.exists():
            print(f'  WARNING: {path.name} not found, skipping')
            continue
        frames.append(load_xls(path))

    if not frames:
        raise FileNotFoundError(f'No files found for prefix "{prefix}"')

    df = pd.concat(frames, ignore_index=True)
    return clean(df, meter_name)

## 2. Load 普通高壓空調 (AC sub-meter)

This is the only direct AC ground-truth. Files: individual 2016–2020 + one combined 2021–2025 file.

In [ ]:
print('Loading 普通高壓空調 (AC meter)...')

# 2016–2020: individual files
ac_frames = []
for yr in range(2016, 2021):
    path = DATA_DIR / f'普通高壓空調{yr}.xls'
    ac_frames.append(load_xls(path))

# 2021–2025: single combined file
path_combined = DATA_DIR / '普通高壓空調2021-2025.xls'
ac_frames.append(load_xls(path_combined))

df_ac = pd.concat(ac_frames, ignore_index=True)
df_ac = clean(df_ac, '普通_AC')

# Remove the duplicate boundary row at 2021-01-01 00:00 from the overlap
df_ac = df_ac[~df_ac.index.duplicated(keep='first')]

print(f'\n普通 AC meter: {len(df_ac):,} hourly rows')
print(f'  Range: {df_ac.index.min()} → {df_ac.index.max()}')
print(f'  Avg power: {df_ac["kw"].mean():.1f} kW')
print(f'  Total energy: {df_ac["kwh"].sum():,.0f} kWh')
df_ac.head(3)

## 3. Load Lecture Hall Total Electricity Meters

In [ ]:
print('Loading 共同教室...')
df_gongtong = load_meter('共同教室', '共同教室')
print(f'  {len(df_gongtong):,} rows | {df_gongtong.index.min().date()} → {df_gongtong.index.max().date()}')
print(f'  Avg power: {df_gongtong["kw"].mean():.1f} kW | Total: {df_gongtong["kwh"].sum():,.0f} kWh')

In [ ]:
print('Loading 博雅 sub-meters (館一, 館二, 三, 四)...')
df_boya1 = load_meter('博雅館一', '博雅館一')
df_boya2 = load_meter('博雅館二', '博雅館二')
df_boya3 = load_meter('博雅三',   '博雅三')
df_boya4 = load_meter('博雅四',   '博雅四')

# Combine all 博雅 sub-meters into one aggregated series
# Align on datetime index, sum kW and kWh
boya_kw  = (df_boya1['kw'].rename('b1') + df_boya2['kw'].rename('b2') +
            df_boya3['kw'].rename('b3') + df_boya4['kw'].rename('b4'))
boya_kwh = (df_boya1['kwh'].rename('b1') + df_boya2['kwh'].rename('b2') +
            df_boya3['kwh'].rename('b3') + df_boya4['kwh'].rename('b4'))

# Keep individual sub-meters for detailed analysis; also expose aggregated
df_boya_all = pd.concat([df_boya1, df_boya2, df_boya3, df_boya4])

for name, df in [('博雅館一', df_boya1), ('博雅館二', df_boya2),
                 ('博雅三',   df_boya3), ('博雅四',   df_boya4)]:
    print(f'  {name}: {len(df):,} rows | avg {df["kw"].mean():.1f} kW | total {df["kwh"].sum():,.0f} kWh')

In [ ]:
print('Loading 新生大樓 (binary XLS)...')
df_xinsheng = load_meter('新生大樓', '新生大樓')
print(f'  {len(df_xinsheng):,} rows | {df_xinsheng.index.min().date()} → {df_xinsheng.index.max().date()}')
print(f'  Avg power: {df_xinsheng["kw"].mean():.1f} kW | Total: {df_xinsheng["kwh"].sum():,.0f} kWh')

## 4. Coverage Summary

In [ ]:
all_meters = {
    '普通_AC':  df_ac,
    '共同教室': df_gongtong,
    '博雅館一': df_boya1,
    '博雅館二': df_boya2,
    '博雅三':   df_boya3,
    '博雅四':   df_boya4,
    '新生大樓': df_xinsheng,
}

summary_rows = []
for name, df in all_meters.items():
    # Expected hourly count for full coverage
    expected = pd.date_range(df.index.min().floor('H'),
                             df.index.max().ceil('H'),  freq='H')
    pct_coverage = len(df) / len(expected) * 100
    summary_rows.append({
        'Meter': name,
        'Start': df.index.min().date(),
        'End':   df.index.max().date(),
        'Rows':  len(df),
        'Coverage %': round(pct_coverage, 1),
        'Avg kW': round(df['kw'].mean(), 1),
        'Total MWh': round(df['kwh'].sum() / 1000, 1),
        'Missing hrs': len(expected) - len(df),
    })

summary = pd.DataFrame(summary_rows).set_index('Meter')
print('=== Data Coverage Summary ===')
summary

## 5. Missing Timestamp Check & Gap Filling

In [ ]:
def fill_gaps(df: pd.DataFrame, meter_name: str,
              max_interp_hours: int = 3) -> pd.DataFrame:
    """
    Reindex to a complete hourly DatetimeIndex.
    Gaps <= max_interp_hours are linearly interpolated.
    Longer gaps are left as NaN and flagged.
    """
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq='H')
    df = df.reindex(full_idx)

    # Mark which rows were originally missing
    df['gap_filled'] = df['kw'].isna()

    # Interpolate only short gaps
    numeric_cols = ['kw', 'kwh', 'pf', 'kva', 'kvar']
    df[numeric_cols] = (
        df[numeric_cols]
        .interpolate(method='linear', limit=max_interp_hours)
    )

    long_gaps = df['kw'].isna().sum()
    filled    = df['gap_filled'].sum() - long_gaps
    print(f'  [{meter_name}] Interpolated {filled} short gaps; '
          f'{long_gaps} long gaps remain as NaN')

    # Re-derive time features for newly inserted rows
    df['year']       = df.index.year
    df['month']      = df.index.month
    df['hour']       = df.index.hour
    df['weekday']    = df.index.weekday
    df['is_weekend'] = df['weekday'] >= 5
    df['season']     = df['month'].map({
        12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring',  4: 'Spring', 5: 'Spring',
        6: 'Summer',  7: 'Summer', 8: 'Summer',
        9: 'Autumn', 10: 'Autumn', 11: 'Autumn'
    })
    df['is_semester'] = df['month'].isin([2, 3, 4, 5, 6, 9, 10, 11, 12])
    df['meter']       = meter_name
    return df


print('Filling gaps (interpolating short gaps ≤ 3 hours)...')
df_ac        = fill_gaps(df_ac,        '普通_AC')
df_gongtong  = fill_gaps(df_gongtong,  '共同教室')
df_boya1     = fill_gaps(df_boya1,     '博雅館一')
df_boya2     = fill_gaps(df_boya2,     '博雅館二')
df_boya3     = fill_gaps(df_boya3,     '博雅三')
df_boya4     = fill_gaps(df_boya4,     '博雅四')
df_xinsheng  = fill_gaps(df_xinsheng,  '新生大樓')

## 6. Quick Validation Plots

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Monthly Average Power (kW) — All Lecture Hall Meters', fontsize=13)

plot_data = [
    ('普通_AC (AC only)', df_ac,       '#e74c3c'),
    ('共同教室 (total)', df_gongtong,  '#3498db'),
    ('博雅 (total, all sub-meters)', df_boya_all, '#2ecc71'),
    ('新生大樓 (total)', df_xinsheng,  '#9b59b6'),
]

for ax, (label, df, color) in zip(axes, plot_data):
    monthly = df['kw'].resample('ME').mean()
    ax.plot(monthly.index, monthly.values, color=color, linewidth=1.2)
    ax.set_ylabel('Avg kW', fontsize=9)
    ax.set_title(label, fontsize=10, loc='left')
    ax.grid(axis='y', alpha=0.3)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('monthly_avg_power.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to monthly_avg_power.png')

In [ ]:
# Annual total kWh per meter
fig, ax = plt.subplots(figsize=(12, 5))

annual_meters = {
    '普通_AC':  df_ac,
    '共同教室': df_gongtong,
    '博雅館一': df_boya1,
    '博雅館二': df_boya2,
    '博雅三':   df_boya3,
    '博雅四':   df_boya4,
    '新生大樓': df_xinsheng,
}

years = range(2016, 2026)
x = np.arange(len(years))
width = 0.12
colors = ['#e74c3c', '#3498db', '#27ae60', '#2980b9', '#1abc9c', '#16a085', '#9b59b6']

for i, (name, df) in enumerate(annual_meters.items()):
    annual = [df[df['year'] == yr]['kwh'].sum() / 1000 for yr in years]  # MWh
    ax.bar(x + i * width, annual, width, label=name, color=colors[i], alpha=0.85)

ax.set_xticks(x + width * 3)
ax.set_xticklabels(years)
ax.set_xlabel('Year')
ax.set_ylabel('Total MWh')
ax.set_title('Annual Electricity Consumption by Meter (2016–2025)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('annual_mwh_by_meter.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export Cleaned Data

Save cleaned DataFrames as Parquet for fast loading in subsequent notebooks.

In [ ]:
import os
os.makedirs('cleaned', exist_ok=True)

export_map = {
    'cleaned/ac_普通.parquet':         df_ac,
    'cleaned/gongtong_共同教室.parquet': df_gongtong,
    'cleaned/boya1_博雅館一.parquet':   df_boya1,
    'cleaned/boya2_博雅館二.parquet':   df_boya2,
    'cleaned/boya3_博雅三.parquet':     df_boya3,
    'cleaned/boya4_博雅四.parquet':     df_boya4,
    'cleaned/xinsheng_新生大樓.parquet': df_xinsheng,
}

for fpath, df in export_map.items():
    # Drop object columns that can't serialize cleanly to parquet
    df_out = df.select_dtypes(exclude='object').copy()
    # Keep meter name as a category column
    df_out['meter'] = df['meter'].astype('category')
    df_out.to_parquet(fpath)
    print(f'Saved {fpath}  ({os.path.getsize(fpath) / 1024:.0f} KB)')

print('\nAll cleaned data exported to cleaned/')

## 8. Data Quality Report

In [ ]:
print('=' * 60)
print('DATA QUALITY REPORT')
print('=' * 60)

for name, df in all_meters.items():
    n_total   = len(df)
    n_missing = df['kw'].isna().sum()
    n_filled  = df.get('gap_filled', pd.Series(False, index=df.index)).sum()
    pf_low    = (df['pf'] < 80).sum() if 'pf' in df.columns else 0
    print(f'\n{name}')
    print(f'  Total rows:          {n_total:>8,}')
    print(f'  Missing kW (NaN):    {n_missing:>8,}  ({n_missing/n_total*100:.2f}%)')
    print(f'  Gap-filled rows:     {n_filled:>8,}')
    print(f'  Low power factor     {pf_low:>8,}  (pf < 80%)')
    print(f'  kW range:            [{df["kw"].min():.1f}, {df["kw"].max():.1f}]')